# SASV: ECAPA + AASIST score-sum

Fair comparison vs published Baseline1-v2 (ECAPA + AASIST ≈ **1.71%** SASV-EER on eval).

```text
s_sasv = s_asv + P_bona(AASIST)
```

- Reuses **ECAPA** `s_asv` from `runs/ecapa_plus_lfcc_{split}/scores_{split}.csv` (notebooks 03/04)
- Scores **AASIST** only on unique test utterances (local `aasist/` clone + `AASIST.pth`)
- Tune / smoke on **dev**; lock **eval** once

Needs: `torch`, `soundfile`, AASIST weights, and your existing LFCC score CSVs.

In [5]:
from pathlib import Path
import json
import sys

ROOT = Path.cwd()
if not (ROOT / "aasist_fusion_lib.py").exists():
    ROOT = Path(r"D:\speaker-verification-system\replay-cnn-baseline\experiments\sasv_la2019")
sys.path.insert(0, str(ROOT))

import torch
from experiment_lib import DEFAULT_LA, DEFAULT_SASV, RUNS_DIR, _REPO_ROOT
from aasist_fusion_lib import DEFAULT_AASIST, score_ecapa_aasist_fusion

print("cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
print("LA:", DEFAULT_LA.exists())
print("SASV:", DEFAULT_SASV.exists())
print("AASIST:", DEFAULT_AASIST.exists(), DEFAULT_AASIST)
print("weights:", (DEFAULT_AASIST / "models" / "weights" / "AASIST.pth").exists())

cuda: True
NVIDIA GeForce RTX 4060 Laptop GPU
LA: True
SASV: True
AASIST: True D:\speaker-verification-system\aasist
weights: True


## Knobs

1. `SMOKE = True` → 500 stratified trials (sanity)
2. `SMOKE = False`, `SPLIT = "dev"` → full dev
3. Then `SPLIT = "eval"` **once** (locked; do not retune)

Requires prior LFCC score CSV for that split (`ecapa_plus_lfcc_{split}/scores_*.csv`).

In [6]:
SMOKE = False
SPLIT = "dev"       # then "eval" once
MAX_TRIALS = 500 if SMOKE else 0
DEVICE = "cuda"
FORCE_CPU = False

ECAPA_CSV = RUNS_DIR / f"ecapa_plus_lfcc_{SPLIT}" / f"scores_{SPLIT}.csv"
print("ECAPA CSV:", ECAPA_CSV.exists(), ECAPA_CSV)

ECAPA CSV: True D:\speaker-verification-system\replay-cnn-baseline\experiments\sasv_la2019\runs\ecapa_plus_lfcc_dev\scores_dev.csv


## Run fusion

In [7]:
summary = score_ecapa_aasist_fusion(
    split=SPLIT,
    max_trials=MAX_TRIALS,
    device=DEVICE,
    force_cpu=FORCE_CPU,
    la_root=DEFAULT_LA,
    sasv_root=DEFAULT_SASV,
    aasist_root=DEFAULT_AASIST,
    ecapa_csv=ECAPA_CSV,
    output_dir=RUNS_DIR / f"ecapa_plus_aasist_{SPLIT}",
)
{
    "system": summary["system"],
    "split": summary["split"],
    "n": summary["num_scored"],
    "unique_utts": summary["num_unique_test_utts"],
    "sasv_eer_%": summary["sasv_eer_percent"],
    "sv_eer_%": summary["sv_eer_percent"],
    "spf_eer_%": summary["spf_eer_percent"],
}

Trials: {'target': 1484, 'nontarget': 5768, 'spoof': 22296, 'total': 29548} | reuse s_asv from scores_dev.csv
AASIST on cuda


AASIST utts:   0%|          | 0/24844 [00:00<?, ?it/s]

{
  "system": "ecapa_plus_aasist_sum",
  "split": "dev",
  "max_trials": 0,
  "num_scored": 29548,
  "key_counts": {
    "target": 1484,
    "nontarget": 5768,
    "spoof": 22296,
    "total": 29548
  },
  "device": "cuda",
  "cm_backend": "aasist",
  "fusion": "s_asv + P_bona(AASIST softmax)",
  "ecapa_csv": "D:\\speaker-verification-system\\replay-cnn-baseline\\experiments\\sasv_la2019\\runs\\ecapa_plus_lfcc_dev\\scores_dev.csv",
  "num_unique_test_utts": 24844,
  "sasv_eer": 0.0074123989222642,
  "sv_eer": 0.013477088948786993,
  "spf_eer": 0.0013006817363945113,
  "sasv_eer_percent": 0.74123989222642,
  "sv_eer_percent": 1.3477088948786993,
  "spf_eer_percent": 0.13006817363945114
}


{'system': 'ecapa_plus_aasist_sum',
 'split': 'dev',
 'n': 29548,
 'unique_utts': 24844,
 'sasv_eer_%': 0.74123989222642,
 'sv_eer_%': 1.3477088948786993,
 'spf_eer_%': 0.13006817363945114}

## Compare with locked systems

In [8]:
print(f"{SPLIT}:")
for label, path in [
    ("ecapa_only", RUNS_DIR / f"ecapa_only_{SPLIT}" / f"metrics_{SPLIT}.json"),
    ("ecapa_plus_lfcc", RUNS_DIR / f"ecapa_plus_lfcc_{SPLIT}" / f"metrics_{SPLIT}.json"),
    ("ecapa_plus_wavlm", RUNS_DIR / f"ecapa_plus_wavlm_{SPLIT}" / f"metrics_{SPLIT}.json"),
    ("ecapa_plus_aasist", RUNS_DIR / f"ecapa_plus_aasist_{SPLIT}" / f"metrics_{SPLIT}.json"),
]:
    if not path.exists():
        print(f"  Missing: {path}")
        continue
    m = json.loads(path.read_text(encoding="utf-8"))
    print(
        f"  {label:20s}  SASV={m['sasv_eer_percent']:.4f}%  "
        f"SV={m['sv_eer_percent']:.4f}%  SPF={m['spf_eer_percent']:.4f}%"
    )

dev:
  ecapa_only            SASV=15.2291%  SV=1.2483%  SPF=17.9090%
  ecapa_plus_lfcc       SASV=1.1438%  SV=2.0978%  SPF=0.0897%
  ecapa_plus_wavlm      SASV=7.3450%  SV=11.8598%  SPF=3.8410%
  ecapa_plus_aasist     SASV=0.7412%  SV=1.3477%  SPF=0.1301%


## Run order

1. `SMOKE = True`, `SPLIT = "dev"` — confirm loop works  
2. `SMOKE = False`, `SPLIT = "dev"` — full dev (~unique utts ≪ 29k trials)  
3. `SMOKE = False`, `SPLIT = "eval"` — locked eval once  

Reference: published B1-v2 ≈ **1.71%** SASV-EER on eval.